# Chat with the instruction-fine-tuned GPT-2
Loads the fine-tuned model: `gpt2-large774M-alpaca-lora.pth` (GPT-2 large + Alpaca + LoRA) if attached, otherwise `gpt2-medium355M-sft.pth`, into the same `GPTModel` built from scratch, and opens a chat window.

STEPS TO RUN

**Before running (Kaggle):**
1. Settings → Accelerator → **GPU T4 x2**, Internet → **On**.
2. Right panel → **Add Input** → **Your Work** → select the notebook whose run produced `gpt2-medium355M-sft.pth`.
3. Run the cells **one by one in an interactive session** (not Save & Run All: the chat window only lives while the session runs).

The last cell prints a public link (`…gradio.live`). Open it in any browser to chat. It works while this session is running.

In [1]:
%pip install -q tiktoken gradio

import glob, torch, torch.nn as nn, tiktoken
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = tiktoken.get_encoding("gpt2")
print("Device:", device)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
Note: you may need to restart the kernel to use updated packages.
Device: cuda


## The GPT model (same architecture as in the project)

In [2]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, num_tokens, _ = x.shape
        # [b, T, d_out] -> [b, num_heads, T, head_dim]
        queries = self.W_query(x).view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        keys = self.W_key(x).view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        values = self.W_value(x).view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)

        attn_scores = queries @ keys.transpose(2, 3)                      # [b, heads, T, T]
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)                   # causal mask before softmax
        attn_weights = torch.softmax(attn_scores / self.head_dim ** 0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context = (attn_weights @ values).transpose(1, 2).contiguous().view(b, num_tokens, self.d_out)
        return self.out_proj(context)


class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        return self.scale * (x - mean) / torch.sqrt(var + self.eps) + self.shift


class GELU(nn.Module):
    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(torch.sqrt(torch.tensor(2.0 / torch.pi)) * (x + 0.044715 * torch.pow(x, 3))))


class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layers(x)


class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(cfg["emb_dim"], cfg["emb_dim"], cfg["context_length"],
                                      cfg["drop_rate"], cfg["n_heads"], cfg["qkv_bias"])
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        shortcut = x                                   # Pre-LN: shortcut taken before LayerNorm
        x = self.drop_shortcut(self.att(self.norm1(x)))
        x = x + shortcut
        shortcut = x
        x = self.drop_shortcut(self.ff(self.norm2(x)))
        return x + shortcut


class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        self.trf_blocks = nn.Sequential(*[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        _, seq_len = in_idx.shape
        x = self.tok_emb(in_idx) + self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        return self.out_head(x)                        # [b, T, vocab_size]


def text_to_token_ids(text, tokenizer):
    return torch.tensor(tokenizer.encode(text, allowed_special={"<|endoftext|>"})).unsqueeze(0)

def token_ids_to_text(token_ids, tokenizer):
    return tokenizer.decode(token_ids.squeeze(0).tolist())

def generate(model, idx, max_new_tokens, context_size, temperature=0.0, top_k=None, eos_id=None):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]              # crop to the supported context length
        with torch.no_grad():
            logits = model(idx_cond)[:, -1, :]         # only the last position
        if top_k is not None:
            top_logits, _ = torch.topk(logits, top_k)
            logits = torch.where(logits < top_logits[:, -1:], torch.full_like(logits, -torch.inf), logits)
        if temperature > 0.0:
            probs = torch.softmax(logits / temperature, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
        else:
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)
        if eos_id is not None and (idx_next == eos_id).all():
            break
        idx = torch.cat((idx, idx_next), dim=1)
    return idx


## Load the fine-tuned weights

In [3]:
import glob
def find(name):
    return glob.glob(f"/kaggle/input/**/{name}", recursive=True) + glob.glob(name)

BASE = {"vocab_size": 50257, "context_length": 1024, "drop_rate": 0.0, "qkv_bias": True}
if find("gpt2-large774M-alpaca-lora.pth"):          # the improved model (GPT-2 large + Alpaca + LoRA)
    SFT_PATH = find("gpt2-large774M-alpaca-lora.pth")[0]
    CFG = {**BASE, "emb_dim": 1280, "n_layers": 36, "n_heads": 20}
elif find("gpt2-medium355M-sft.pth"):               # the first model (GPT-2 medium + ~935 examples)
    SFT_PATH = find("gpt2-medium355M-sft.pth")[0]
    CFG = {**BASE, "emb_dim": 1024, "n_layers": 24, "n_heads": 16}
else:
    raise FileNotFoundError("No fine-tuned model found: add your training notebook's output with Add Input")

model = GPTModel(CFG)
model.load_state_dict(torch.load(SFT_PATH, map_location="cpu"))   # half-precision files are converted automatically
model.to(device).eval()
print("Loaded", SFT_PATH)

def format_input(instruction, input_text=""):
    text = ("Below is an instruction that describes a task. "
            "Write a response that appropriately completes the request."
            f"\n\n### Instruction:\n{instruction}")
    return text + (f"\n\n### Input:\n{input_text}" if input_text.strip() else "")

def generate_better(model, idx, max_new_tokens, context_size, temperature=0.0, top_k=50,
                    repetition_penalty=1.3, no_repeat_ngram=3, eos_id=50256):
    """Like generate(), plus two standard fixes for repetition loops:
    a repetition penalty (tokens already used become less likely) and blocking any 3-word phrase from repeating."""
    prompt_len = idx.shape[1]
    for _ in range(max_new_tokens):
        with torch.no_grad():
            logits = model(idx[:, -context_size:])[:, -1, :].float()
        generated = idx[0, prompt_len:].tolist()
        if generated and repetition_penalty != 1.0:
            used = torch.tensor(sorted(set(generated)), device=logits.device)
            vals = logits[0, used]
            logits[0, used] = torch.where(vals > 0, vals / repetition_penalty, vals * repetition_penalty)
        n = no_repeat_ngram
        if n and len(generated) >= n - 1:
            prefix = tuple(generated[-(n - 1):])
            banned = {generated[i + n - 1] for i in range(len(generated) - n + 1) if tuple(generated[i:i + n - 1]) == prefix}
            if banned:
                logits[0, list(banned)] = -torch.inf
        if temperature > 0:
            top_vals, _ = torch.topk(logits, top_k)
            logits = torch.where(logits < top_vals[:, -1:], torch.full_like(logits, -torch.inf), logits)
            idx_next = torch.multinomial(torch.softmax(logits / temperature, dim=-1), num_samples=1)
        else:
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)
        if idx_next.item() == eos_id:
            break
        idx = torch.cat((idx, idx_next), dim=1)
    return idx

def respond(instruction, input_text="", temperature=0.0, max_new_tokens=200):
    prompt = format_input(instruction, input_text) + "\n\n### Response:\n"   # must match training
    idx = text_to_token_ids(prompt, tokenizer).to(device)
    out = generate_better(model, idx, max_new_tokens=int(max_new_tokens), context_size=CFG["context_length"],
                          temperature=float(temperature))
    return token_ids_to_text(out, tokenizer)[len(prompt):].strip()

print(respond("Write a short poem about the ocean."))

Loaded /kaggle/input/notebooks/ekansh0801/notebookb367a134e8/gpt2-large774M-alpaca-lora.pth
The ocean, vast and deep
A place of peace and serenity,
Where the waves crash against shore,
And the sun shines down on us all. 

It's a sea that never sleeps,
Its tides rise and fall,
As we sail through its depths,
With gentle currents to guide our way.


## Chat window
Each message is treated as one instruction, the way the model was fine-tuned (it was not trained on multi-turn conversations, so it does not remember earlier messages).
Optionally put extra text in **Input** (for example, a sentence to rewrite).

In [4]:
import gradio as gr

try:
    demo.close()          # close a previous chat window, if one is running
except Exception:
    pass

def chat(message, history, input_text=None, temperature=None, max_new_tokens=None):
    input_text = input_text or ""
    temperature = 0.0 if temperature is None else float(temperature)
    max_new_tokens = 200 if max_new_tokens is None else int(max_new_tokens)
    return respond(message, input_text, temperature, max_new_tokens)

demo = gr.ChatInterface(
    fn=chat,
    title="GPT-2, built from scratch and instruction fine-tuned",
    description="Type an instruction, e.g. 'Give three synonyms for happy' or 'Convert 45 kilometers to meters'.",
    additional_inputs=[
        gr.Textbox(label="Input (optional)", placeholder="Extra text the instruction refers to"),
        gr.Slider(0.0, 1.5, value=0.0, step=0.1, label="Temperature (0 = most likely answer)"),
        gr.Slider(20, 400, value=200, step=10, label="Max new tokens"),
    ],
    examples=[["Convert the active sentence to passive voice: 'The chef cooks the meal every day.'", "", 0.0, 200],
              ["What is the capital of France?", "", 0.0, 200],
              ["Rewrite the sentence in a formal tone.", "Hey, can you send me the report asap?", 0.0, 200]],
)
demo.launch(share=True, debug=False)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://b461a38dcb317ce8e6.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
